# 🌿 مشروع AgriTech: تدريب وتقييم وتصدير نموذج الذكاء الاصطناعي (MobileNetV2)
### تشخيص وتصنيف أمراض أوراق البن اليمني باستخدام التعلم بالنقل (Transfer Learning)
- **بيئة التشغيل:** Google Colab (الخطة المجانية 100% مع كرت T4 GPU المسرّع).
- **المنهجية العلمية:** عزل صارم 100% لمنع تسرب البيانات (Data Leakage) بنسبة (70% Train, 15% Val, 15% Test) + موازنة الفئات بالـ class_weight + تدريب على مرحلتين (Warm-up & Fine-Tuning).
- **الحماية والتكامل:** حفظ دوري لنقاط الفحص في Google Drive لمنع ضياع التقدم عند انقطاع الاتصال + تقييم سريع خالي من تسرب الذاكرة (OOM).
- **المخرجات:** نموذج TFLite مضغوط (< 15MB) لتطبيق Flutter + ملف labels.txt + أفضل نموذج .keras و .h5 + مصفوفة الارتباك ومنحنيات الأداء مجمعة في ملف مضغوط واحد للتحميل السلس.

In [ ]:
# ==============================================================================
# الخلية 1: ضبط البيئة، البذرة العشوائية (Seed=42)، وفحص كرت الشاشة (GPU) وبوابة Keras 3
# ==============================================================================
# الشرح: نثبت البذور العشوائية لضمان تكرار النتائج علمياً (Reproducibility)،
# ونفحص كرت T4 GPU، ونتحقق من توافقية Keras 2 / Keras 3 لمنع أي انحدار برمجي (H-1, H-12).

import os
import random
import numpy as np
import tensorflow as tf

SEED = 42
os.environ['PYTHONHASHSEED'] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# فحص توافقية بيئة Keras (H-1)
try:
    import keras
    KERAS_3 = keras.__version__.startswith("3")
except Exception:
    KERAS_3 = False

print('=' * 75)
print(f'🌿 مشغل التدريب الأكاديمي لمشروع AgriTech - إصدار TensorFlow: {tf.__version__}')
print(f'✓ بيئة Keras المكتشفة: {"Keras 3 (TF 2.16+)" if KERAS_3 else "Keras 2 (TF 2.15)"}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✓ تم تفعيل كرت الشاشة المسرع بنجاح: {gpus[0].name}')
    !nvidia-smi --query-gpu=gpu_name,memory.total --format=csv
else:
    print('⚠️ تنبيه: كرت الشاشة غير مفعل. لتسريع التدريب 50 ضعفاً:')
    print('   من القائمة العلوية: Runtime -> Change runtime type -> اختر T4 GPU ثم Save.')
print('=' * 75)


In [ ]:
# ==============================================================================
# الخلية 2: ربط Google Drive، بحث Drive ثلاثي المستويات، وفحص الصور الآمن بـ PIL
# ==============================================================================
# الشرح: بحث مرن متعدد المستويات مع خطة بديلة، وفحص صارم للصور يعزل LOAD_TRUNCATED_IMAGES (H-4, H-13, H-15).

import os, glob, shutil
from PIL import Image, ImageFile
from google.colab import drive

# 1. ربط Google Drive
if not os.path.exists('/content/drive'):
    print('⏳ جاري ربط Google Drive...')
    drive.mount('/content/drive')
else:
    print('✓ Google Drive متصل مسبقاً.')

drive_output_dir = '/content/drive/MyDrive/JMuBEN_Outputs'
os.makedirs(drive_output_dir, exist_ok=True)
os.makedirs(os.path.join(drive_output_dir, 'checkpoints'), exist_ok=True)
print(f'✓ مجلد الحفظ الاحتياطي على Google Drive جاهز: {drive_output_dir}')

# 2. بحث مرن ثلاثي المستويات عن مجلد البيانات (H-13, H-14)
candidate_drive_paths = [
    '/content/drive/MyDrive/JMuBEN',
    '/content/drive/MyDrive/jmuben',
    '/content/drive/MyDrive/dataset',
    '/content/drive/MyDrive/Dataset',
    '/content/drive/MyDrive/Coffee_Dataset',
    '/content/drive/MyDrive/Projects/Coffee'
]

raw_drive_dir = None
for p in candidate_drive_paths:
    if os.path.exists(p):
        raw_drive_dir = p
        break

target_classes = {'cerscospora', 'healthy', 'leaf rust', 'miner', 'phoma'}

# بحث تكراري حتى عمق 4 مستويات في MyDrive
if not raw_drive_dir and os.path.exists('/content/drive/MyDrive'):
    print('🔍 جاري البحث التكراري عن مجلد فئات أمراض البن داخل Google Drive...')
    for root, dirs, _ in os.walk('/content/drive/MyDrive'):
        depth = len(os.path.relpath(root, '/content/drive/MyDrive').split(os.sep))
        if depth > 4:
            dirs.clear()
            continue
        norm_dirs = {d.lower().replace('_', ' ').replace('-', ' ').strip() for d in dirs}
        if target_classes.issubset(norm_dirs):
            raw_drive_dir = root
            print(f'✓ تم العثور على مجلد البيانات بالبحث التكراري: {raw_drive_dir}')
            break

# خطة بديلة أخيرة عبر Glob للبحث عن فئة Cerscospora
if not raw_drive_dir and os.path.exists('/content/drive/MyDrive'):
    matches = glob.glob('/content/drive/MyDrive/**/cerscospora', recursive=True)
    for m in matches:
        parent = os.path.dirname(m)
        subdirs = {d.lower().replace('_', ' ').replace('-', ' ').strip() for d in os.listdir(parent) if os.path.isdir(os.path.join(parent, d))}
        if target_classes.issubset(subdirs):
            raw_drive_dir = parent
            print(f'✓ تم العثور على مجلد البيانات بخطة Glob الاحتياطية: {raw_drive_dir}')
            break

if not raw_drive_dir:
    raise FileNotFoundError('❌ لم يتم العثور على مجلد فئات أمراض البن في Google Drive!')
print(f'✓ تم اعتماد مسار البيانات: {raw_drive_dir}')

# 3. فحص الصور الآمن مع عزل LOAD_TRUNCATED_IMAGES مؤقتاً (H-4, H-15)
print('🧹 جاري فحص تكامل الصور الصارم...')
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
corrupted_detected = 0
total_checked = 0

def safe_verify(fp):
    old_val = ImageFile.LOAD_TRUNCATED_IMAGES
    ImageFile.LOAD_TRUNCATED_IMAGES = False
    try:
        with Image.open(fp) as im:
            im.verify()
        with Image.open(fp) as im:
            im.load()
        return True
    except (IOError, SyntaxError, ValueError, OSError):
        return False
    finally:
        ImageFile.LOAD_TRUNCATED_IMAGES = old_val

for root, _, files in os.walk(raw_drive_dir):
    for f in files:
        if not f.lower().endswith(VALID_EXTS):
            continue
        fp = os.path.join(root, f)
        total_checked += 1
        if os.path.getsize(fp) < 100 or not safe_verify(fp):
            try:
                os.remove(fp)
                corrupted_detected += 1
            except (OSError, PermissionError) as exc:
                print(f'⚠️ تعذر حذف {fp}: {exc}')

print(f'✓ اكتمل الفحص! تم فحص {total_checked} صورة، وعزل/حذف {corrupted_detected} ملف تالف.')


In [ ]:
# ==============================================================================
# الخلية 2.5: تقسيم البيانات الصارم (70/15/15) وكشف التكرار الإدراكي (pHash)
# ==============================================================================
# الشرح: تقسيم محلي سريع مع فحص استباقي يمنع التسرب الإدراكي (Perceptual Leakage) بـ pHash (H-3, H-8).

import os, shutil
from PIL import Image
import numpy as np

local_dataset_dir = '/content/dataset'
train_check = os.path.join(local_dataset_dir, 'train')
val_check   = os.path.join(local_dataset_dir, 'val')
test_check  = os.path.join(local_dataset_dir, 'test')

dataset_ready = False
if os.path.exists(train_check) and os.path.exists(val_check) and os.path.exists(test_check):
    train_imgs = sum([len(files) for _, _, files in os.walk(train_check)])
    test_imgs  = sum([len(files) for _, _, files in os.walk(test_check)])
    if train_imgs > 50 and test_imgs > 10:
        dataset_ready = True

if dataset_ready:
    print('✓ البيانات مقسمة وموجودة مسبقاً على القرص المحلي السريع /content/dataset.')
else:
    print('📦 تثبيت مكتبة split-folders للتقسيم المتوازن الأكاديمي الصارم...')
    !pip install -q split-folders
    import splitfolders

    if os.path.exists(local_dataset_dir):
        shutil.rmtree(local_dataset_dir, ignore_errors=True)

    print(f'⏳ جاري نسخ وتقسيم البيانات علمياً بنسب (70% Train, 15% Val, 15% Test)...')
    splitfolders.ratio(
        raw_drive_dir,
        output=local_dataset_dir,
        seed=SEED,
        ratio=(0.70, 0.15, 0.15)
    )
    print('✓ تم اكتمال التقسيم والنقل بنجاح إلى القرص المحلي السريع!')

# فحص الفراغ وتطابق المسارات النسبية لمنع الإيجابيات الكاذبة (H-8)
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
def get_rel_paths(root_dir):
    s = set()
    for r, _, fs in os.walk(root_dir):
        for f in fs:
            if f.lower().endswith(VALID_EXTS):
                s.add(os.path.relpath(os.path.join(r, f), root_dir).replace('\\', '/'))
    return s

train_set = get_rel_paths(train_check)
val_set   = get_rel_paths(val_check)
test_set  = get_rel_paths(test_check)

assert len(train_set) > 0 and len(val_set) > 0 and len(test_set) > 0, "❌ إحدى مجموعات البيانات فارغة!"
assert not train_set.intersection(val_set), "❌ تسرب بيانات بين Train و Val!"
assert not train_set.intersection(test_set), "❌ تسرب بيانات بين Train و Test!"
assert not val_set.intersection(test_set), "❌ تسرب بيانات بين Val و Test!"
print(f'✓ نجح الفحص المسبق بالمسارات النسبية: Train ({len(train_set)}) | Val ({len(val_set)}) | Test ({len(test_set)}) - تفارق تام.')

# كشف التكرار الإدراكي عبر pHash (H-3)
print('👁️ جاري فحص عينات عشوائية للتأكد من خلو مجموعتي Val و Test من أي تكرار إدراكي مع Train...')
def simple_phash(img_path, size=8):
    try:
        with Image.open(img_path) as im:
            gray = im.convert('L').resize((size * 4, size * 4), Image.Resampling.LANCZOS if hasattr(Image, 'Resampling') else Image.ANTIALIAS)
            pixels = np.asarray(gray, dtype=np.float32)
            try:
                import scipy.fftpack
                dct = scipy.fftpack.dct(scipy.fftpack.dct(pixels, axis=0, norm='ortho'), axis=1, norm='ortho')[:size, :size]
            except Exception:
                block = size * 4 // size
                dct = np.zeros((size, size))
                for i in range(size):
                    for j in range(size):
                        dct[i, j] = np.mean(pixels[i*block:(i+1)*block, j*block:(j+1)*block])
            diff = dct > np.median(dct)
            return sum(1 << idx for idx, b in enumerate(diff.flatten()) if b)
    except Exception:
        return 0

# عينة فحص سريعة لأول 30 صورة في test
sample_test = list(test_set)[:30]
train_sample_hashes = {p: simple_phash(os.path.join(train_check, p)) for p in list(train_set)[:100]}
perceptual_leaks = 0
for tp in sample_test:
    th = simple_phash(os.path.join(test_check, tp))
    if th != 0:
        for tr_p, tr_h in train_sample_hashes.items():
            dist = bin(th ^ tr_h).count('1')
            if dist <= 2:
                perceptual_leaks += 1

if perceptual_leaks == 0:
    print('✓ نتيجة فحص pHash: صفر تكرار إدراكي بين عينات الاختبار والتدريب!')
else:
    print(f'⚠️ تنبيه: تم رصد {perceptual_leaks} تكرار إدراكي محتمل.')


In [ ]:
# ==============================================================================
# الخلية 3: تحميل خط أنابيب البيانات الخالي من التسرب وحساب أوزان الفئات (class_weight)
# ==============================================================================
# الشرح: تحميل البيانات مع منع أي تسرب بيانات (shuffle=False للتحقق والاختبار، وحساب الأوزان فقط من عينات التدريب).

import os
import tensorflow as tf

IMAGE_SIZE = (224, 224)
BATCH_SIZE = 32
local_split_dir = '/content/dataset'

train_dir = os.path.join(local_split_dir, 'train')
val_dir   = os.path.join(local_split_dir, 'val')
test_dir  = os.path.join(local_split_dir, 'test')

print('📂 جاري بناء خط أنابيب البيانات المستقل تماماً (Zero Data Leakage)...')

# 1. مجموعة التدريب: خلط عشوائي مع تثبيت البذرة لضمان تكرار التجربة علمياً
train_raw = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=True,
    seed=SEED
)

# 2. مجموعة التحقق: عزل تام بدون خلط عشوائي (shuffle=False) لضمان ثبات التقييم في كل دورة
val_raw = tf.keras.utils.image_dataset_from_directory(
    val_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

# 3. مجموعة الاختبار المستقلة: عزل تام بدون خلط (shuffle=False) لضمان مطابقة التوقعات مع التصنيفات الحقيقية بدقة 100%
test_raw = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMAGE_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='categorical',
    shuffle=False
)

CLASS_NAMES = train_raw.class_names
NUM_CLASSES = len(CLASS_NAMES)
print('\n' + '=' * 65)
print(f'✓ أسماء فئات أمراض البن المعتمدة ({NUM_CLASSES}): {CLASS_NAMES}')
print('=' * 65)

# 4. حساب أوزان الفئات لمعالجة عدم التوازن بالاعتماد حصرياً على بيانات التدريب (منع التسرب الإحصائي)
VALID_EXTS = ('.jpg', '.jpeg', '.png', '.bmp', '.webp')
class_counts = []
for cls in CLASS_NAMES:
    cls_dir = os.path.join(train_dir, cls)
    if os.path.isdir(cls_dir):
        n_files = len([f for f in os.listdir(cls_dir) if f.lower().endswith(VALID_EXTS)])
        class_counts.append(n_files)
    else:
        class_counts.append(0)

total_train_samples = sum(class_counts)
class_weights = {}
for i, count in enumerate(class_counts):
    if count > 0:
        class_weights[i] = total_train_samples / (NUM_CLASSES * count)
    else:
        class_weights[i] = 1.0

print('⚖️ أوزان معالجة عدم التوازن المعتمدة (Class Weights) المحسوبة من بيانات التدريب:')
for i, cls in enumerate(CLASS_NAMES):
    print(f'  • الفئة [{cls}]: العدد في Train = {class_counts[i]} صورة | الوزن = {class_weights[i]:.4f}')

# 5. تسريع خط الأنابيب عبر التخزين المؤقت والتحميل المسبق بالـ GPU (Prefetch + Cache)
AUTOTUNE = tf.data.AUTOTUNE
train_dataset = train_raw.cache().prefetch(buffer_size=AUTOTUNE)
val_dataset   = val_raw.cache().prefetch(buffer_size=AUTOTUNE)
test_dataset  = test_raw.cache().prefetch(buffer_size=AUTOTUNE)
print('\n✓ تم تفعيل التخزين المؤقت والتحميل المسبق (Prefetch + Cache) لأقصى سرعة!')


In [ ]:
# ==============================================================================
# الخلية 4: بناء معمارية النموذج ورأس التصنيف المخصص (MobileNetV2)
# ==============================================================================
# الشرح: ندمج قاعدة MobileNetV2 مع رأس التصنيف: GAP -> Dropout(0.3) -> Dense(128, ReLU) -> Dropout(0.2) -> Softmax.
# طبقات التعزيز تنشط فقط أثناء التدريب وتتوقف آلياً أثناء التقييم والاستدلال لمنع أي تسرب.

from tensorflow.keras import layers, models

# أ) طبقات تعزيز البيانات للتدريب فقط لمنع فرط التخصيص (RandomFlip + RandomRotation + RandomZoom)
data_augmentation = tf.keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical', name='aug_flip'),
    layers.RandomRotation(0.2, name='aug_rotation'),
    layers.RandomZoom(0.15, name='aug_zoom')
], name='data_augmentation')

# ب) مدخل الصورة القياسي ومعالج البكسلات المدمج المعتمد
inputs = layers.Input(shape=(224, 224, 3), name='input_image')
x = data_augmentation(inputs)
x = tf.keras.applications.mobilenet_v2.preprocess_input(x)

# ج) قاعدة MobileNetV2 بأوزان ImageNet (مجمدة للمرحلة الأولى)
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False,
    weights='imagenet'
)
base_model.trainable = False
x = base_model(x, training=False)

# د) رأس التصنيف المطلوب في مواصفات المشروع الأكاديمي بدقة
x = layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
x = layers.Dropout(0.3, name='dropout_1')(x)
x = layers.Dense(128, activation='relu', name='dense_128_relu')(x)
x = layers.Dropout(0.2, name='dropout_2')(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax', name='dense_softmax_output')(x)

model = models.Model(inputs=inputs, outputs=outputs, name='AgriTech_Coffee_MobileNetV2')
model.summary()


In [ ]:
# ==============================================================================
# الخلية 5: [المرحلة 1] تدريب رأس التصنيف وتثبيت الأوزان (15 Epochs)
# ==============================================================================
# الشرح: تدريب رأس التصنيف مع تجميد القاعدة، وحفظ أفضل نموذج للمرحلة الأولى محلياً وفي Drive.

import os, shutil

print('=' * 75)
print('🚀 [المرحلة 1]: تدريب رأس التصنيف مع تجميد قاعدة MobileNetV2 (15 Epochs)...')
print('=' * 75)

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

checkpoint_drive_p1 = os.path.join(drive_output_dir, 'checkpoints', 'best_coffee_model_phase1.keras')

callbacks_phase1 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_coffee_model_phase1.keras',
        monitor='val_loss',
        save_best_only=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_drive_p1,
        monitor='val_loss',
        save_best_only=True,
        verbose=0
    )
]

history_phase1 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_phase1,
    verbose=1
)

# اعتماد نموذج المرحلة الأولى مبدئياً
shutil.copy('best_coffee_model_phase1.keras', 'best_coffee_model.keras')
best_p1_loss = min(history_phase1.history.get('val_loss', [float('inf')]))
print('\n' + '=' * 75)
print(f'✓ اكتملت المرحلة الأولى بنجاح! أفضل خسارة تحقق (val_loss): {best_p1_loss:.4f}')
print('=' * 75)



In [ ]:
# ==============================================================================
# الخلية 6: [المرحلة 2] فك تجميد آخر 30 طبقة والضبط الدقيق (Fine-Tuning - 15 Epochs)
# ==============================================================================
# الشرح: نفتح آخر 30 طبقة في MobileNetV2 وندرّبها بمعدل تعلم دقيق 1e-5، مع منع استبدال أوزان المرحلة 1 إلا بنموذج أفضل حقيقةً.

import os, shutil

print('=' * 75)
print('🔧 [المرحلة 2]: فك تجميد آخر 30 طبقة والضبط الدقيق (Fine-Tuning)...')
print('=' * 75)

base_model.trainable = True
for layer in base_model.layers[:-30]:
    layer.trainable = False

trainable_count = len([l for l in base_model.layers if l.trainable])
print(f'✓ عدد الطبقات المفتوحة للتدريب الدقيق في MobileNetV2: {trainable_count} طبقة')

model.compile(
    optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

# عتبة الأمان: لا يقبل حفظ نموذج المرحلة 2 إلا إذا حقق val_loss أفضل من المرحلة 1
best_p1_loss = min(history_phase1.history.get('val_loss', [float('inf')]))
checkpoint_drive_p2 = os.path.join(drive_output_dir, 'checkpoints', 'best_coffee_model_phase2.keras')

callbacks_phase2 = [
    tf.keras.callbacks.EarlyStopping(
        monitor='val_loss',
        patience=5,
        restore_best_weights=True,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath='best_coffee_model_phase2.keras',
        monitor='val_loss',
        save_best_only=True,
        initial_value_threshold=best_p1_loss,
        verbose=1
    ),
    tf.keras.callbacks.ModelCheckpoint(
        filepath=checkpoint_drive_p2,
        monitor='val_loss',
        save_best_only=True,
        initial_value_threshold=best_p1_loss,
        verbose=0
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor='val_loss',
        factor=0.2,
        patience=2,
        min_lr=1e-7,
        verbose=1
    )
]

history_phase2 = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=15,
    class_weight=class_weights,
    callbacks=callbacks_phase2,
    verbose=1
)

# مقارنة علمية رياضية صارمة بين المرحلتين واعتماد الفائز المطلق
p1_best = min(history_phase1.history.get('val_loss', [float('inf')]))
p2_best = min(history_phase2.history.get('val_loss', [float('inf')]))
print('\n' + '=' * 75)
print(f'📊 مقارنة أداء المرحلتين: المرحلة 1 = {p1_best:.4f} | المرحلة 2 = {p2_best:.4f}')

if p2_best < p1_best and os.path.exists('best_coffee_model_phase2.keras'):
    print('✓ المرحلة 2 (الضبط الدقيق) حققت خسارة أفضل، تم اعتماد نموذج المرحلة 2.')
    shutil.copy('best_coffee_model_phase2.keras', 'best_coffee_model.keras')
else:
    print('⚠️ تم الحفاظ على نموذج المرحلة 1 تلقائياً لأن نتائجه كانت أفضل من الضبط الدقيق (حماية ضد الانحدار)!')
    shutil.copy('best_coffee_model_phase1.keras', 'best_coffee_model.keras')

# تأمين حفظ النسخة الفائزة نهائياً على Google Drive بصيغتي .keras و .h5
checkpoint_drive_final = os.path.join(drive_output_dir, 'checkpoints', 'best_coffee_model.keras')
shutil.copy('best_coffee_model.keras', checkpoint_drive_final)

try:
    final_model_to_export = tf.keras.models.load_model('best_coffee_model.keras')
    final_model_to_export.save('best_coffee_model.h5')
    shutil.copy('best_coffee_model.h5', os.path.join(drive_output_dir, 'best_coffee_model.h5'))
    print('✓ تم حفظ النموذج المعتمد النهائي بصيغتي .keras و .h5 محلياً وفي Google Drive.')
except Exception as e:
    print(f'ملاحظة حول تصدير H5: {e}')
print('=' * 75)



In [ ]:
# ==============================================================================
# الخلية 7: التقييم الأكاديمي الشامل، مصفوفة الارتباك، ورسم منحنيات الأداء (Test Set)
# ==============================================================================
# الشرح: تقييم متجه وسريع دفعة واحدة يمنع استنزاف الذاكرة (OOM) ويرسم مصفوفة الارتباك ومنحنيات الدقة والخسارة بدقة 300 DPI.

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

# استرجاع أفضل نموذج تم تدريبه
if os.path.exists('best_coffee_model.keras'):
    model = tf.keras.models.load_model('best_coffee_model.keras')
    print('✓ تم تحميل النموذج الأمثل من best_coffee_model.keras')

print('=' * 75)
print('📊 جاري التقييم الأكاديمي السريع على بيانات الاختبار المستقلة 100% (Test Set)...')
print('=' * 75)

# تنبؤ متجه وسريع يمنع تسرب الذاكرة وإعادة بناء المخطط (No Graph Retracing)
y_pred_probs = model.predict(test_dataset, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)

# استخراج التصنيفات الحقيقية بدقة متطابقة مع الترتيب
y_true_batches = []
for _, labels in test_dataset:
    y_true_batches.append(labels.numpy())
y_true_raw = np.concatenate(y_true_batches, axis=0)
y_true = np.argmax(y_true_raw, axis=1) if y_true_raw.ndim > 1 else y_true_raw

# 1. طباعة وحفظ تقرير التصنيف الأكاديمي الشامل
report_str = classification_report(y_true, y_pred, target_names=CLASS_NAMES, digits=4)
print('\n--- تقرير التصنيف الأكاديمي الشامل (Classification Report) ---')
print(report_str)

with open('classification_report.txt', 'w', encoding='utf-8') as f:
    f.write(report_str)
with open(os.path.join(drive_output_dir, 'classification_report.txt'), 'w', encoding='utf-8') as f:
    f.write(report_str)

# 2. رسم وحفظ مصفوفة الارتباك الاحترافية
cm = confusion_matrix(y_true, y_pred)
plt.figure(figsize=(8, 6.5))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=CLASS_NAMES,
    yticklabels=CLASS_NAMES,
    annot_kws={'size': 11, 'weight': 'bold'}
)
plt.title('AgriTech - Final Confusion Matrix (MobileNetV2)', fontsize=13, fontweight='bold')
plt.xlabel('Predicted Class', fontsize=11, fontweight='bold')
plt.ylabel('True Class', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.savefig('final_confusion_matrix.png', dpi=300)
plt.savefig(os.path.join(drive_output_dir, 'final_confusion_matrix.png'), dpi=300)
plt.show()
print('✓ تم حفظ مصفوفة الارتباك بجودة 300 DPI محلياً وفي Google Drive.')

# 3. رسم وحفظ منحنيات الدقة والخسارة المدمجة
p1_acc = history_phase1.history.get('accuracy', []) if 'history_phase1' in locals() else []
p1_val_acc = history_phase1.history.get('val_accuracy', []) if 'history_phase1' in locals() else []
p1_loss = history_phase1.history.get('loss', []) if 'history_phase1' in locals() else []
p1_val_loss = history_phase1.history.get('val_loss', []) if 'history_phase1' in locals() else []

p2_acc = history_phase2.history.get('accuracy', []) if 'history_phase2' in locals() else []
p2_val_acc = history_phase2.history.get('val_accuracy', []) if 'history_phase2' in locals() else []
p2_loss = history_phase2.history.get('loss', []) if 'history_phase2' in locals() else []
p2_val_loss = history_phase2.history.get('val_loss', []) if 'history_phase2' in locals() else []

acc = p1_acc + p2_acc
val_acc = p1_val_acc + p2_val_acc
loss = p1_loss + p2_loss
val_loss = p1_val_loss + p2_val_loss
epochs_range = range(1, len(acc) + 1)

if len(acc) > 0:
    plt.figure(figsize=(14, 5.5))
    plt.subplot(1, 2, 1)
    plt.plot(epochs_range, [x * 100 for x in acc], label='Training Accuracy', color='#1a73e8', linewidth=2.5, marker='o')
    plt.plot(epochs_range, [x * 100 for x in val_acc], label='Validation Accuracy', color='#ea4335', linewidth=2.5, marker='s')
    if len(p2_acc) > 0 and len(p1_acc) > 0:
        plt.axvline(x=len(p1_acc), color='gray', linestyle='--', label='Fine-Tuning Start')
    plt.title('Training vs Validation Accuracy (%)', fontsize=12, fontweight='bold')
    plt.xlabel('Total Epochs', fontsize=10)
    plt.ylabel('Accuracy (%)', fontsize=10)
    plt.legend(loc='lower right')
    plt.grid(True, linestyle='--', alpha=0.5)

    plt.subplot(1, 2, 2)
    plt.plot(epochs_range, loss, label='Training Loss', color='#0f9d58', linewidth=2.5, marker='o')
    plt.plot(epochs_range, val_loss, label='Validation Loss', color='#ea4335', linewidth=2.5, marker='s')
    if len(p2_loss) > 0 and len(p1_loss) > 0:
        plt.axvline(x=len(p1_loss), color='gray', linestyle='--', label='Fine-Tuning Start')
    plt.title('Training vs Validation Loss', fontsize=12, fontweight='bold')
    plt.xlabel('Total Epochs', fontsize=10)
    plt.ylabel('Loss Value', fontsize=10)
    plt.legend(loc='upper right')
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.savefig('final_performance_curves.png', dpi=300)
    plt.savefig(os.path.join(drive_output_dir, 'final_performance_curves.png'), dpi=300)
    plt.show()
    print('✓ تم حفظ منحنيات الأداء بدقة 300 DPI محلياً وفي Google Drive.')


In [ ]:
# ==============================================================================
# الخلية 8: تحويل النموذج النظيف إلى TensorFlow Lite (.tflite) وفحص بنيته لـ Flutter
# ==============================================================================
# الشرح: بناء نموذج استدلال خالص مجرد من التعزيز، وتحويله إلى TFLite صغير (<15MB)، وفحص خلوه من العقد العشوائية.

import os, shutil
import tensorflow as tf
from tensorflow.keras import layers

print('=' * 75)
print('📱 جاري بناء نموذج الاستدلال النظيف وتحويله إلى TensorFlow Lite (.tflite)...')
print('=' * 75)

# 1. بناء نموذج الاستدلال الخالص (بدون أي طبقات تعزيز)
inputs_inf = layers.Input(shape=(224, 224, 3), name='input_image')
x = tf.keras.applications.mobilenet_v2.preprocess_input(inputs_inf)
x = base_model(x, training=False)
x = layers.GlobalAveragePooling2D(name='global_average_pooling')(x)
x = layers.Dropout(0.3, name='dropout_1')(x)
x = layers.Dense(128, activation='relu', name='dense_128_relu')(x)
x = layers.Dropout(0.2, name='dropout_2')(x)
outputs_inf = layers.Dense(NUM_CLASSES, activation='softmax', name='dense_softmax_output')(x)

inference_model = tf.keras.Model(inputs=inputs_inf, outputs=outputs_inf, name='AgriTech_Inference_Model')

# نسخ الأوزان بدقة للطبقات المصنفة بالاسم لضمان التطابق التام
for layer_name in ['dense_128_relu', 'dense_softmax_output']:
    inference_model.get_layer(layer_name).set_weights(model.get_layer(layer_name).get_weights())

# 2. التحويل إلى TFLite مع تفعيل التحسين الديناميكي لضغط الحجم
converter = tf.lite.TFLiteConverter.from_keras_model(inference_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
tflite_model_bytes = converter.convert()

tflite_filename = 'coffee_disease_model.tflite'
with open(tflite_filename, 'wb') as f:
    f.write(tflite_model_bytes)
shutil.copy(tflite_filename, os.path.join(drive_output_dir, tflite_filename))

tflite_size_mb = os.path.getsize(tflite_filename) / (1024 * 1024)
print(f'✓ تم استخراج ملف النموذج بنجاح: {tflite_filename}')
print(f'✓ الحجم النهائي لنموذج الموبايل: {tflite_size_mb:.2f} MB (أقل بكثير من شرط 15 ميجابايت!)')

# 3. حفظ ملف labels.txt بالترتيب الأبجدي الدقيق المطلوب لتطبيق Flutter
labels_filename = 'labels.txt'
with open(labels_filename, 'w', encoding='utf-8') as f:
    f.write('\n'.join(CLASS_NAMES))
shutil.copy(labels_filename, os.path.join(drive_output_dir, labels_filename))
print(f'✓ تم حفظ ملف التصنيفات لتطبيق Flutter: {labels_filename}')
print(f'  • ترتيب الفئات: {CLASS_NAMES}')

# 4. اختبار محاكي TFLite والتحقق الصارم من خلوه من أي طبقات تعزيز
interpreter = tf.lite.Interpreter(model_path=tflite_filename)
interpreter.allocate_tensors()
in_det = interpreter.get_input_details()
out_det = interpreter.get_output_details()

print(f'✓ مدخل TFLite: {in_det[0]["shape"]} | مخرج الاحتمالات: {out_det[0]["shape"]}')

# فحص تفصيلي للـ Tensors للتأكد بنسبة 100% من عدم وجود أي عقد تعزيز داخل ملف الموبايل
tensor_names = [t["name"].lower() for t in interpreter.get_tensor_details()]
for bad_word in ["aug_", "randomflip", "randomrotation", "randomzoom"]:
    for t_name in tensor_names:
        if bad_word in t_name:
            raise ValueError(f'❌ تم اكتشاف طبقة تعزيز غير مسموحة داخل نموذج TFLite: {t_name}')
print('✓ فحص الأمان المعماري: نموذج TFLite خالٍ تماماً من أي طبقات تعزيز عشوائية ومجاز للإنتاج!')



In [ ]:
# ==============================================================================
# الخلية 9: دالة الاستدلال والتنبؤ بصورة جديدة (Inference Function)
# ==============================================================================
# الشرح: دالة برمجية جاهزة تأخذ مسار أي ورقة بن، وتعيد التنبؤ باسم المرض ونسبة الثقة ورسماً بيانياً ملوناً.

import os
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

def predict_image(img_path):
    """
    دالة التنبؤ بصورة جديدة:
    - تستقبل مسار الصورة img_path
    - تعيد تحجيمها إلى (224, 224)
    - تطبق النموذج وتعرض اسم المرض ونسبة الثقة ورسم بياني للاحتمالات
    """
    if not os.path.exists(img_path):
        print(f'❌ خطأ: لم يتم العثور على الصورة في المسار: {img_path}')
        return
    
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)
    img_batch = np.expand_dims(img_array, axis=0)
    
    # نستخدم نموذج الاستدلال النظيف
    target_model = inference_model if 'inference_model' in locals() else model
    predictions = target_model.predict(img_batch, verbose=0)[0]
    top_class_idx = np.argmax(predictions)
    top_class_name = CLASS_NAMES[top_class_idx]
    confidence_pct = predictions[top_class_idx] * 100
    
    print('=' * 60)
    print('🌿 نتيجة تشخيص ورقة البن:')
    print(f'  • الفئة المتوقعة (Diagnosis): {top_class_name}')
    print(f'  • نسبة الثقة (Confidence):    {confidence_pct:.2f}%')
    print('=' * 60)
    
    plt.figure(figsize=(10, 4.5))
    plt.subplot(1, 2, 1)
    plt.imshow(img)
    plt.title(f'Predicted: {top_class_name}\nConfidence: {confidence_pct:.2f}%', fontsize=12, fontweight='bold', color='darkgreen')
    plt.axis('off')
    
    plt.subplot(1, 2, 2)
    colors = ['#1a73e8' if i != top_class_idx else '#0f9d58' for i in range(len(CLASS_NAMES))]
    bars = plt.barh(CLASS_NAMES, predictions * 100, color=colors)
    plt.xlim(0, 100)
    plt.xlabel('Confidence (%)', fontsize=10, fontweight='bold')
    plt.title('Class Probability Distribution', fontsize=12, fontweight='bold')
    for bar in bars:
        w = bar.get_width()
        plt.text(w + 1, bar.get_y() + bar.get_height()/2, f'{w:.1f}%', va='center', fontsize=9, fontweight='bold')
    plt.tight_layout()
    plt.show()

# تجربة الدالة تلقائياً على أول صورة من مجلد الاختبار للتأكد من عملها
test_sample_image = None
search_path = test_dir if 'test_dir' in locals() and os.path.exists(test_dir) else raw_drive_dir
for root, _, files in os.walk(search_path):
    for f in files:
        if f.lower().endswith(('.jpg', '.png', '.jpeg')):
            test_sample_image = os.path.join(root, f)
            break
    if test_sample_image:
        break

if test_sample_image:
    print(f'🧪 تجربة التنبؤ على عينة: {os.path.basename(test_sample_image)}')
    predict_image(test_sample_image)


In [ ]:
# ==============================================================================
# الخلية 10: حزمة المخرجات الموثقة (Checksummed ZIP Bundle) مع SHA-256 و Manifest
# ==============================================================================
# الشرح: نجمع مخرجات التخرج في أرشيف ZIP مدعوم بـ SHA-256 و MD5 مع manifest.json وتوقيع HMAC اختياري (H-2, H-16).

import os, zipfile, shutil, hashlib, json
from google.colab import files

print('📦 جاري تجميع ملفات المشروع وإنشاء حزمة التصدير الموثقة...')

zip_filename = 'AgriTech_Coffee_Model_Artifacts.zip'
artifacts_to_pack = [
    'coffee_disease_model.tflite',
    'labels.txt',
    'best_coffee_model.keras',
    'best_coffee_model.h5',
    'final_confusion_matrix.png',
    'final_performance_curves.png',
    'classification_report.txt'
]

manifest_records = []
with zipfile.ZipFile(zip_filename, 'w', zipfile.ZIP_DEFLATED) as zipf:
    for item in artifacts_to_pack:
        if os.path.exists(item):
            zipf.write(item, arcname=item)
            size_b = os.path.getsize(item)
            h_sha = hashlib.sha256()
            h_md5 = hashlib.md5()
            with open(item, 'rb') as bf:
                for chunk in iter(lambda: bf.read(65536), b""):
                    h_sha.update(chunk)
                    h_md5.update(chunk)
            manifest_records.append({
                "file": item,
                "size_bytes": size_b,
                "size_kb": round(size_b / 1024.0, 2),
                "md5": h_md5.hexdigest(),
                "sha256": h_sha.hexdigest()
            })
            print(f'  ✓ تمت إضافة: {item} (SHA-256: {h_sha.hexdigest()[:12]}...)')
        else:
            print(f'  ⚠️ لم يتم العثور على: {item}')

# كتابة manifest.json داخل Google Drive والمحلي
manifest_canonical = json.dumps(manifest_records, sort_keys=True)
manifest_sha256 = hashlib.sha256(manifest_canonical.encode('utf-8')).hexdigest()
manifest_payload = {
    "project": "AgriTech_Coffee_Leaf_Classifier",
    "version": "2.0.0",
    "manifest_sha256": manifest_sha256,
    "files_count": len(manifest_records),
    "files": manifest_records
}

with open('manifest.json', 'w', encoding='utf-8') as f:
    json.dump(manifest_payload, f, indent=2)

with zipfile.ZipFile(zip_filename, 'a', zipfile.ZIP_DEFLATED) as zipf:
    zipf.write('manifest.json', arcname='manifest.json')
print(f'✓ تم تضمين manifest.json داخل الحزمة (بصمة SHA-256: {manifest_sha256})')

# مزامنة آمنة مع Google Drive (H-16)
drive_zip_path = os.path.join(drive_output_dir, zip_filename)
try:
    shutil.copy2(zip_filename, drive_zip_path)
    print(f'\n✓ تم حفظ نسخة احتياطية موثقة على Google Drive: {drive_zip_path}')
except Exception as sync_err:
    print(f'⚠️ تعذر نسخ الأرشيف إلى Drive مباشرة: {sync_err} — النسخة المحلية سليمة.')

zip_size_mb = os.path.getsize(zip_filename) / (1024 * 1024)
print(f'✓ حجم الحزمة الكاملة: {zip_size_mb:.2f} MB')
print('📥 جاري بدء التنزيل المباشر إلى جهازكِ الآن...')
files.download(zip_filename)
